# Acceptance: summer2-style output requests

**This is a sign-off page, not a tutorial.** It checks the one-call output requests on
`OutputSet` (plan `plans/output-requests.plan.md`, roadmap step 51).

summer2 asked for each derived output in one call. summer4's `OutputSet` can express all of them,
but a port wrote its own helpers for the common cases. Requests are those helpers, built into
`OutputSet`. Each is one assignment, and its docstring shows the expression it assigns:

| summer2 | summer4 request | Stands for |
|---|---|---|
| `request_output_for_compartments(n, comps, strata=)` | `out.compartments(n, where)` | `Compartments().select(where).sum_by()` |
| `request_output_for_flow(n, f, source_strata=, dest_strata=)` | `out.flow(n, f, source=, dest=)` | `FlowMass(f).select(Source(..) & Dest(..)).sum_by()` |
| `request_aggregate_output(n, sources)` | `out.aggregate(n, sources)` | `ref(a) + ref(b) + …` |
| `request_cumulative_output(n, src, start_time=)` | `out.cumulative(n, src, start=)` | `ref(src).cumulative_integral(start=)` |
| `request_function_output(n, Function(f, args))` | `out.function(n, f, *args)` | `defer(f)(refs and params…)` |
| `save_results=False` | `keep=False` on any request or `out.set` | the name is computed but not returned |
| a `for` loop per stratum | `by=[age, reach, (age, reach)]` | one `sum_by` per grouping, a row per stratum |
| `model.run(params)` then `get_derived_outputs_df()` | `cm.run(params, …, outputs=out)` | `out.evaluate(cm.run(…, save=out.plan()), params)` |

Nothing here asserts; the checks are in `tests/test_output_requests.py`. Run every cell top to
bottom (under a minute). Each section ends with a **What to check** box.

| | You are checking that… |
|---|---|
| A | a request gives the same series as the expression it stands for |
| B | `by=` adds one named output per stratum, and the strata add up to the total |
| C | `keep=False` names are computed but not returned |
| D | requests mix with raw expressions, ufuncs and chained post-ops |
| E | fan-out's traced program grows no faster per stratum than a select per stratum |

In [ ]:
import jax
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowMass,
    FlowModel,
    OutputSet,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    Source,
    TransitionFlow,
)

state = Property("state", ("S", "I", "R"))
age = Property("age", ("child", "adult", "elder"))
reach = Property("reach", ("near", "far"))
pmap = PropertyMap.from_property(state).stratify(age).stratify(reach)


def build(pmap):
    """An SIR in every stratum: a per-capita infection rate, then recovery."""
    model = FlowModel(pmap)
    model.add_flow(TransitionFlow("infection", state["S"], state["I"], Param("beta")))
    model.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("gamma")))
    return model.compile()


cm = build(pmap)
params = {"beta": 0.08, "gamma": 0.1}
y0 = PropertyData.wrap(
    pmap, np.where(pmap.kleene(state["S"]) == 1, np.linspace(200.0, 900.0, pmap.size), 0.0)
)
RUN = dict(t0=0.0, t1=60.0, dt=1.0)
pmap.size

## A. A request is the expression it stands for

The next cell asks for four outputs with requests, solving once with `run(outputs=)`. The cell
after it builds the same four by hand and solves the long way (`plan`, `run`, `evaluate`). The
figure draws both versions of the incidence among elders, and the table gives the largest
difference between the two versions of each output.

In [ ]:
requested = OutputSet()
requested.compartments("infectious", state["I"])
requested.flow("incidence_elder", "infection", source=age["elder"])
requested.aggregate("everyone_recovering", ["infectious"])
requested.cumulative("cumulative_incidence_elder", "incidence_elder")
named = cm.run(params, y0, outputs=requested, **RUN)

by_hand = OutputSet()
by_hand["infectious"] = Compartments().select(state["I"]).sum_by()
by_hand["incidence_elder"] = FlowMass("infection").select(Source(age["elder"])).sum_by()
by_hand["everyone_recovering"] = by_hand.ref("infectious")
by_hand["cumulative_incidence_elder"] = by_hand.ref("incidence_elder").cumulative_integral()
hand = by_hand.evaluate(cm.run(params, y0, save=by_hand.plan(), **RUN), params)

days = pd.Index(np.asarray(named["infectious"].times.values), name="day")
pd.DataFrame(
    {
        "request: out.flow(..., source=age['elder'])": np.asarray(named["incidence_elder"].array()),
        "by hand: FlowMass(...).select(Source(...)).sum_by()": np.asarray(
            hand["incidence_elder"].array()
        ),
    },
    index=days,
).plot(
    title="A. Incidence among elders, requested and by hand",
    labels={"value": "people per day", "variable": ""},
).show()
pd.Series(
    {n: float(np.max(np.abs(named[n].array() - hand[n].array()))) for n in requested.kept()},
    name="largest difference",
)

> **What to check (A).** The two lines are one line. Every difference in the table is 0.

## B. One output per stratum with `by=`

`by=[age, reach, (age, reach)]` adds an output for every age, every reach stratum and every
(age, reach) pair, named the summer2-port way (`prevalenceXage_child`,
`prevalenceXage_childXreach_far`). The figure stacks the three age outputs and draws the total
over them.

In [ ]:
fan = OutputSet()
fan.compartments("prevalence", state["I"], by=[age, reach, (age, reach)])
print(fan.kept())
named_b = cm.run(params, y0, outputs=fan, **RUN)

frame_b = pd.DataFrame(
    {n: np.asarray(named_b[n].array()) for n in fan.kept() if n.count("X") == 1 and "age" in n},
    index=days,
)
figure = frame_b.plot.area(
    title="B. Infectious people by age, stacked, with the total",
    labels={"value": "people", "variable": ""},
)
figure.add_scatter(
    x=days, y=np.asarray(named_b["prevalence"].array()), name="prevalence (total)",
    line=dict(color="black", dash="dash"),
)
figure.show()

> **What to check (B).** The printed names are `prevalence`, then three `Xage_` names, two
> `Xreach_` names and six `Xage_…Xreach_` names. The dashed total runs along the top of the
> stacked areas.

## C. Intermediates: `keep=False`

summer2's `save_results=False` becomes `keep=False`. The population and the infectious count
below are intermediates: prevalence reads them, but they are not returned, and `posterior_runs`
would not record them. `keys()` still lists every name; `kept()` lists what `evaluate` returns.
`subset(names)` makes the same choice after the fact.

In [ ]:
inter = OutputSet()
inter.compartments("population", keep=False)
inter.compartments("infectious", state["I"], keep=False)
inter["percent_infectious"] = 100 * inter.ref("infectious") / inter.ref("population")
named_c = cm.run(params, y0, outputs=inter, **RUN)

named_c["percent_infectious"].to_pandas().rename(columns={"value": "% infectious"}).plot(
    title="C. Percent infectious, from two intermediates",
    labels={"value": "% of population", "variable": ""},
).show()
pd.DataFrame(
    {
        "keys()": [", ".join(inter.keys())],
        "kept()": [", ".join(inter.kept())],
        "returned by run(outputs=)": [", ".join(named_c.keys())],
        "subset(['population']).kept()": [", ".join(inter.subset(["population"]).kept())],
    }
).T

> **What to check (C).** The curve rises from about 0 and falls again. Only `percent_infectious`
> is kept and returned; `keys()` lists all three names.

## D. Escape hatches

A request returns its expression, so post-ops chain on it, and plain assignments, `ref`, NumPy
functions and `defer` all mix with requests in one set. The figure draws a request, a 7-day
rolling mean chained on it, and the log of the request through `np.log`.

In [ ]:
mixed = OutputSet()
incidence = mixed.flow("incidence", "infection")
mixed["incidence_7day"] = incidence.rolling(7)
mixed["log_incidence"] = np.log(mixed.ref("incidence"))
mixed["incidence_per_beta"] = mixed.ref("incidence") / Param("beta")
named_d = cm.run(params, y0, outputs=mixed, **RUN)

pd.DataFrame(
    {
        "incidence (request)": np.asarray(named_d["incidence"].array()),
        "incidence.rolling(7) (chained)": np.asarray(named_d["incidence_7day"].array()),
        "exp(np.log(ref)) (ufunc)": np.exp(np.asarray(named_d["log_incidence"].array())),
    },
    index=days,
).plot(
    title="D. A request, a chained post-op and a ufunc in one set",
    labels={"value": "people per day", "variable": ""},
).show()

> **What to check (D).** The request and `exp(log(...))` are the same line; the rolling mean is
> a smoother version of it that starts at day 0 (a short window at the start).

## E. What fan-out costs in the traced program

Each stratum of a fan-out is a row of one grouped sum (a small array), not a fresh select over
the whole compartment save. Every named output is still its own step when `evaluate` is traced,
so the program grows with the number of names either way. The next cell builds models with more
and more age bands, asks for one output per (age, reach) pair both ways, and counts the equations
in the program `jax.make_jaxpr` traces for `evaluate`.

In [ ]:
def jaxpr_size(n_ages, use_by):
    ages = Property("age", tuple(f"a{i}" for i in range(n_ages)))
    p = PropertyMap.from_property(state).stratify(ages).stratify(reach)
    model = build(p)
    out = OutputSet()
    if use_by:
        out.compartments("p", state["I"], by=[(ages, reach)])
    else:
        for a in ages.traits:
            for r in reach.traits:
                out[f"pXage_{a}Xreach_{r}"] = (
                    Compartments().select(state["I"] & ages[a] & reach[r]).sum_by()
                )
    y = PropertyData.wrap(p, np.ones(p.size))
    result = model.run(params, y, save=out.plan(), t0=0.0, t1=5.0, dt=1.0, solver="euler")
    names = [n for n in out.kept() if "X" in n]
    return len(jax.make_jaxpr(lambda r: out.evaluate(r, params, names=names))(result).eqns)


sizes = pd.DataFrame(
    {
        "by=[(age, reach)]": [jaxpr_size(n, True) for n in (2, 4, 8, 16)],
        "a select per stratum": [jaxpr_size(n, False) for n in (2, 4, 8, 16)],
    },
    index=pd.Index([2, 4, 8, 16], name="age bands"),
)
sizes.plot.bar(
    barmode="group",
    title="E. Equations traced to evaluate one output per (age, reach) pair",
    labels={"value": "jaxpr equations", "variable": ""},
).show()
sizes

> **What to check (E).** Both grow by the same number of equations per stratum (the table's
> differences between rows match). The `by=` bars are taller by the same small constant at every
> size: the one grouped sum. Fan-out saves the loop, not equations.